# ✈️ Trafic aérien européen 2025 : fiabilité des données et journées perturbées

**Auteur :** Valentin Rouquier · **Données :** EUROCONTROL, trafic IFR quotidien de 331 aéroports (via Kaggle) · **Outils :** BigQuery (SQL), Python (pandas, matplotlib)

## 🎯 Deux questions

1. **Fiabilité** : quand deux sources comptent les mêmes vols, sont-elles d'accord ? Où, quand, de combien divergent-elles ?
2. **Perturbations** : quelles journées s'écartent anormalement du trafic attendu, et à quels événements correspondent-elles ?

## 🗂️ Organisation du notebook

| Partie | Contenu |
|---|---|
| 0 | Connexion à BigQuery (et reconstruction complète du pipeline si besoin) |
| 1 | Couverture des deux sources |
| 2 | Rapprochement des sources |
| 3 | Détection des journées perturbées |
| 4 | Explication des perturbations et effet réseau |
| 5 | Conclusions et limites |

> **Répartition SQL / Python** : le nettoyage, les contrôles et la détection sont faits en **SQL** (fichiers `sql/01` à `06`, exécutés dans BigQuery). Python sert à **lire les résultats, les visualiser et les interpréter**.

## 0. Connexion à BigQuery

In [ ]:
from google.colab import auth
auth.authenticate_user()   # fenêtre de connexion Google

import io
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROJET  = "eco-flying-nice-vr"
DATASET = "airports_traffic"
client  = bigquery.Client(project=PROJET, location="EU")

def lire(requete: str) -> pd.DataFrame:
    """Exécute une requête BigQuery et renvoie un DataFrame."""
    return client.query(requete).to_dataframe()

def executer(requete: str) -> None:
    """Exécute une requête qui ne renvoie rien (CREATE VIEW...) et attend la fin."""
    client.query(requete).result()

In [ ]:
RECONSTRUIRE = False

# Copie des fichiers sql/ du projet (dans l'ordre : chaque vue dépend de la précédente)
VUES_SQL = {
    "01_staging.sql": """
-- 01_staging.sql
-- Objectif : une table propre, typée et renommée, SANS supprimer de lignes.
-- Les valeurs absentes de la source n°2 restent NULL (absence d'information ≠ zéro vol).

CREATE OR REPLACE VIEW `eco-flying-nice-vr.airports_traffic.stg_airport_traffic` AS
SELECT
  DATE(FLT_DATE)                    AS flight_date,
  APT_ICAO                          AS airport_icao,
  APT_NAME                          AS airport_name,
  STATE_NAME                        AS country,

  -- Source n°1 (toujours renseignée)
  FLT_DEP_1                         AS src1_dep,
  FLT_ARR_1                         AS src1_arr,
  FLT_TOT_1                         AS src1_total,

  -- Source n°2 (renseignée pour une partie des aéroports seulement)
  CAST(FLT_DEP_IFR_2 AS INT64)      AS src2_dep,
  CAST(FLT_ARR_IFR_2 AS INT64)      AS src2_arr,
  CAST(FLT_TOT_IFR_2 AS INT64)      AS src2_total
FROM `eco-flying-nice-vr.airports_traffic.raw_airport_traffic_2025`;
""",
    "03_rapprochement.sql": """
-- 03_rapprochement.sql
-- Question : quand les deux sources existent, sont-elles d'accord ?
-- SAFE_DIVIDE renvoie NULL au lieu de planter si on divise par zéro.

CREATE OR REPLACE VIEW `eco-flying-nice-vr.airports_traffic.rapprochement` AS
SELECT
  flight_date,
  airport_icao,
  airport_name,
  country,
  src1_total,
  src2_total,
  src2_total - src1_total                                  AS ecart,
  SAFE_DIVIDE(src2_total - src1_total, src1_total)         AS ecart_pct,
  CASE
    WHEN src2_total IS NULL                                         THEN 'source 2 absente'
    WHEN ABS(SAFE_DIVIDE(src2_total - src1_total, src1_total)) > 0.05 THEN 'écart > 5 %'
    WHEN src2_total <> src1_total                                    THEN 'écart ≤ 5 %'
    ELSE 'identique'
  END                                                      AS statut
FROM `eco-flying-nice-vr.airports_traffic.stg_airport_traffic`;

-- Synthèse rapide à lancer ensuite :
-- SELECT statut, COUNT(*) AS nb_lignes
-- FROM `eco-flying-nice-vr.airports_traffic.rapprochement`
-- GROUP BY statut ORDER BY nb_lignes DESC;
""",
    "04_jours_perturbes.sql": """
-- 04_jours_perturbes.sql  (version 2 : référence centrée)
-- Question : quels jours s'écartent anormalement du trafic habituel ?
--
-- Principe : on compare chaque jour aux MÊMES jours de la semaine
-- situés 1 à 4 semaines AVANT et APRÈS (8 « voisins » au maximum).
-- Pourquoi avant ET après ? La v1 ne regardait que les 8 semaines précédentes :
-- fin octobre, elle comparait l'hiver IATA à l'été et voyait des « anomalies »
-- partout (simple changement de programme). Une référence centrée corrige ça.
--
-- Le JOIN de la table sur elle-même (« self-join ») sert à aller chercher ces voisins.
-- z_score = (trafic du jour - moyenne des voisins) / écart-type des voisins.
-- En dessous de -3 : journée très inhabituelle → probablement un événement.

CREATE OR REPLACE VIEW `eco-flying-nice-vr.airports_traffic.jours_perturbes` AS
WITH voisins AS (
  SELECT
    jour.flight_date,
    jour.airport_icao,
    jour.airport_name,
    jour.country,
    jour.src1_total,
    AVG(voisin.src1_total)    AS moy_voisins,
    STDDEV(voisin.src1_total) AS ecart_type_voisins,
    COUNT(*)                  AS nb_voisins
  FROM `eco-flying-nice-vr.airports_traffic.stg_airport_traffic` AS jour
  JOIN `eco-flying-nice-vr.airports_traffic.stg_airport_traffic` AS voisin
    ON  voisin.airport_icao = jour.airport_icao
    AND ABS(DATE_DIFF(voisin.flight_date, jour.flight_date, DAY)) BETWEEN 7 AND 28
    AND MOD(DATE_DIFF(voisin.flight_date, jour.flight_date, DAY), 7) = 0   -- même jour de semaine
  GROUP BY jour.flight_date, jour.airport_icao, jour.airport_name, jour.country, jour.src1_total
)
SELECT
  flight_date,
  airport_icao,
  airport_name,
  country,
  src1_total,
  ROUND(moy_voisins, 1)                                             AS moy_voisins,
  ROUND(SAFE_DIVIDE(src1_total - moy_voisins, ecart_type_voisins), 2) AS z_score
FROM voisins
WHERE nb_voisins >= 4          -- pas de verdict sans un minimum de comparaisons
  AND moy_voisins >= 20        -- on ignore les très petits terrains, trop instables
  AND SAFE_DIVIDE(src1_total - moy_voisins, ecart_type_voisins) < -3;
""",
    "05_jours_perturbes_annotes.sql": """
-- 05_jours_perturbes_annotes.sql
-- Objectif : rattacher chaque jour perturbé (vue 04) à un événement connu
-- (table evenements_2025, remplie à la main à partir de recherches sourcées).
--
-- Deux jointures, car un événement peut être :
--   - propre à un pays (ex. grève en France)       → jointure sur date + pays
--   - commun à plusieurs pays (ex. Noël)           → jointure sur date + pays = 'Plusieurs'
-- COALESCE garde l'explication la plus précise (le pays) en priorité.

CREATE OR REPLACE VIEW `eco-flying-nice-vr.airports_traffic.jours_perturbes_annotes` AS
SELECT
  jp.*,
  COALESCE(ev_pays.categorie, ev_multi.categorie, 'Non expliqué') AS categorie,
  COALESCE(ev_pays.evenement, ev_multi.evenement)                 AS evenement,
  COALESCE(ev_pays.source_url, ev_multi.source_url)               AS source_url
FROM `eco-flying-nice-vr.airports_traffic.jours_perturbes` AS jp
LEFT JOIN `eco-flying-nice-vr.airports_traffic.evenements_2025` AS ev_pays
  ON  ev_pays.date = jp.flight_date
  AND ev_pays.pays = jp.country
LEFT JOIN `eco-flying-nice-vr.airports_traffic.evenements_2025` AS ev_multi
  ON  ev_multi.date = jp.flight_date
  AND ev_multi.pays = 'Plusieurs';

-- Synthèse : quelle part des anomalies est expliquée ?
-- SELECT categorie, COUNT(*) AS nb_jours_aeroport,
--        ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct
-- FROM `eco-flying-nice-vr.airports_traffic.jours_perturbes_annotes`
-- GROUP BY categorie
-- ORDER BY nb_jours_aeroport DESC;
""",
    "06_portee_perturbations.sql": """
-- 06_portee_perturbations.sql
-- Question : une anomalie touche-t-elle tout un pays (perturbation « réseau »)
-- ou un seul aéroport (incident « local ») ?
--
-- COUNT(*) OVER (PARTITION BY ...) compte, SANS regrouper les lignes,
-- combien d'aéroports du même pays sont perturbés le même jour.
-- Seuil retenu : 3 aéroports ou plus = réseau (choix simple, à justifier dans le README).

CREATE OR REPLACE VIEW `eco-flying-nice-vr.airports_traffic.jours_perturbes_portee` AS
SELECT
  *,
  COUNT(*) OVER (PARTITION BY flight_date, country)          AS nb_aeroports_pays_jour,
  CASE
    WHEN COUNT(*) OVER (PARTITION BY flight_date, country) >= 3 THEN 'Réseau'
    ELSE 'Local'
  END                                                        AS portee,
  ROUND(SAFE_DIVIDE(src1_total, moy_voisins) * 100 - 100, 1) AS variation_pct
FROM `eco-flying-nice-vr.airports_traffic.jours_perturbes_annotes`;

-- Synthèse portée × catégorie :
-- SELECT portee, categorie, COUNT(*) AS nb
-- FROM `eco-flying-nice-vr.airports_traffic.jours_perturbes_portee`
-- GROUP BY portee, categorie
-- ORDER BY portee, nb DESC;

-- Pistes d'enquête : perturbations réseau encore non expliquées
-- SELECT flight_date, country, ANY_VALUE(nb_aeroports_pays_jour) AS nb_aeroports,
--        STRING_AGG(airport_name, ', ') AS aeroports
-- FROM `eco-flying-nice-vr.airports_traffic.jours_perturbes_portee`
-- WHERE portee = 'Réseau' AND categorie = 'Non expliqué'
-- GROUP BY flight_date, country
-- ORDER BY flight_date;
""",
}

def charger_csv(contenu: bytes, table: str, separateur: str) -> None:
    config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        field_delimiter=separateur,
        skip_leading_rows=1,
        autodetect=True,
        write_disposition="WRITE_TRUNCATE",   # écrase la table si elle existe
    )
    job = client.load_table_from_file(io.BytesIO(contenu), f"{PROJET}.{DATASET}.{table}", job_config=config)
    job.result()
    print(f"✅ {table} : {job.output_rows} lignes chargées")

if RECONSTRUIRE:
    from google.colab import files
    client.create_dataset(f"{PROJET}.{DATASET}", exists_ok=True)

    print("1/2 — Sélectionne airport_traffic_2025.csv")
    f = files.upload()
    charger_csv(list(f.values())[0], "raw_airport_traffic_2025", ",")

    print("2/2 — Sélectionne evenements_2025_v2.csv")
    f = files.upload()
    charger_csv(list(f.values())[0], "evenements_2025", ";")

    for nom, requete in VUES_SQL.items():
        executer(requete)
        print(f"✅ {nom}")

### 0.2 Contrôle : ce qui existe dans le dataset

In [ ]:
for t in client.list_tables(f"{PROJET}.{DATASET}"):
    print(f"{t.table_id:30} ({t.table_type})")

### 0.3 Chargement des données et style des graphiques

In [ ]:
stg = lire(f"SELECT * FROM `{PROJET}.{DATASET}.stg_airport_traffic`")
rap = lire(f"SELECT * FROM `{PROJET}.{DATASET}.rapprochement`")
jpp = lire(f"SELECT * FROM `{PROJET}.{DATASET}.jours_perturbes_portee`")

# BigQuery renvoie les dates dans un type spécial : conversion pour pandas / matplotlib
for df_ in (stg, rap, jpp):
    df_["flight_date"] = pd.to_datetime(df_["flight_date"])

print(f"stg : {stg.shape} · rapprochement : {rap.shape} · jours perturbés : {jpp.shape}")
print(f"Période : {stg['flight_date'].min():%d/%m/%Y} → {stg['flight_date'].max():%d/%m/%Y} · {stg['airport_icao'].nunique()} aéroports")

In [ ]:
BLEU, ORANGE, GRIS = "#2a78d6", "#eb6834", "#8a8984"
ENCRE, ENCRE_2 = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": ENCRE_2,
    "xtick.color": ENCRE_2, "ytick.color": ENCRE_2,
    "axes.grid": True, "grid.color": "#ebeae6", "grid.linewidth": 0.8,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})

## 1. Couverture des deux sources

Le fichier contient deux comptages des mêmes vols :
- **source n°1** (`FLT_*_1`) : renseignée pour 100 % des lignes ;
- **source n°2** (`FLT_*_IFR_2`) : renseignée pour une partie des aéroports seulement.

⚠️ **Correction par rapport à la v1 de ce projet** : la v1 remplaçait les valeurs absentes de la source n°2 par 0, puis supprimait ces lignes comme « hors trafic commercial » (70 % des données perdues). Or **une valeur absente ne veut pas dire zéro vol** : elle signifie que la source n°2 n'a rien transmis. La v2 conserve toutes les lignes.

Requête : `sql/02_couverture_sources.sql`

In [ ]:
couverture = lire(f"""
-- 02_couverture_sources.sql
-- Question : pour chaque aéroport, sur combien de jours la source n°2 est-elle disponible ?
-- On s'attend à trois profils : toujours (≈365 j), jamais (0 j), ou partiel (à expliquer).

SELECT
  airport_icao,
  airport_name,
  country,
  COUNT(*)                                       AS nb_jours,
  COUNTIF(src2_total IS NOT NULL)                AS nb_jours_src2,
  ROUND(COUNTIF(src2_total IS NOT NULL) / COUNT(*) * 100, 1) AS pct_couverture_src2,
  CASE
    WHEN COUNTIF(src2_total IS NOT NULL) = 0        THEN 'jamais'
    WHEN COUNTIF(src2_total IS NOT NULL) = COUNT(*) THEN 'toujours'
    ELSE 'partiel'
  END                                            AS profil_couverture
FROM `{PROJET}.{DATASET}.stg_airport_traffic`
GROUP BY airport_icao, airport_name, country
ORDER BY pct_couverture_src2 DESC, nb_jours DESC;
""")

print(couverture["profil_couverture"].value_counts().to_string())
couverture[couverture["profil_couverture"] == "partiel"]

**Lecture** : 234 aéroports n'ont jamais la source n°2, 86 l'ont toujours, 11 l'ont partiellement (les trois aéroports géorgiens environ 63 jours, Glasgow et Aberdeen 207 jours).

## 2. Rapprochement des deux sources

Sur les jours où les deux sources existent, on calcule l'écart et on classe chaque ligne.

Requête : `sql/03_rapprochement.sql`

In [ ]:
synth_rap = (rap["statut"].value_counts()
             .rename_axis("statut").reset_index(name="nb_lignes"))
synth_rap["pct_hors_absentes"] = (
    synth_rap["nb_lignes"]
    / synth_rap.loc[synth_rap["statut"] != "source 2 absente", "nb_lignes"].sum() * 100
).round(1).where(synth_rap["statut"] != "source 2 absente")
synth_rap

In [ ]:
deux_sources = rap.dropna(subset=["src2_total"])
print(f"Source 2 inférieure à source 1 : {(deux_sources['ecart'] < 0).mean():.0%} des jours")
print(f"Source 2 supérieure à source 1 : {(deux_sources['ecart'] > 0).mean():.0%} des jours")

top_ecarts = (deux_sources.assign(gros_ecart=deux_sources["statut"] == "écart > 5 %")
              .groupby(["airport_icao", "airport_name"])
              .agg(nb_jours=("ecart", "size"),
                   ecart_median_pct=("ecart_pct", lambda s: round(s.median() * 100, 1)),
                   part_jours_gros_ecart=("gros_ecart", lambda s: round(s.mean() * 100, 1)))
              .sort_values("part_jours_gros_ecart", ascending=False))
top_ecarts.head(10)

### Toulouse face à Lyon et Nice

Si l'écart est **constant**, c'est une différence de définition entre les sources. S'il apparaît **par moments**, c'est un événement ou un trou dans les données.

In [ ]:
aeroports = {"LFBO": "Toulouse", "LFLL": "Lyon", "LFMN": "Nice"}

fig, axes = plt.subplots(len(aeroports), 1, figsize=(12, 9), sharex=True, sharey=True)
for ax, (icao, nom) in zip(axes, aeroports.items()):
    d = rap[rap["airport_icao"] == icao].sort_values("flight_date")
    ax.plot(d["flight_date"], d["ecart_pct"] * 100, color=BLEU, linewidth=1)
    ax.axhline(0, color=ENCRE_2, linewidth=0.6)
    for seuil in (-5, 5):
        ax.axhline(seuil, color=ORANGE, linestyle="--", linewidth=0.8)
    ax.set_title(f"{nom} ({icao}) : écart source 2 vs source 1")
    ax.set_ylabel("écart (%)")
axes[0].text(0.995, 0.92, "seuil ±5 %", transform=axes[0].transAxes, ha="right", color=ORANGE, fontsize=9)
plt.tight_layout()
plt.show()

## 3. Détection des journées perturbées

### Méthode

Chaque jour est comparé aux **mêmes jours de la semaine, 1 à 4 semaines avant et après** (un lundi se compare à des lundis). On calcule un **z-score** :

> z = (trafic du jour − moyenne des voisins) / écart-type des voisins

En dessous de **−3**, le jour est jugé anormal. Les très petits terrains (moins de 20 vols par jour en moyenne) sont écartés.

### Une erreur utile

La première version comparait chaque jour aux 8 semaines **précédentes** uniquement. Elle signalait surtout fin octobre et début novembre comme anormaux : c'était en réalité le **passage à l'horaire d'hiver IATA**, une baisse programmée et non une perturbation. La référence **centrée** (avant et après) corrige ce biais.

Requête : `sql/04_jours_perturbes.sql`

In [ ]:
print(f"{len(jpp)} jours-aéroport perturbés détectés")
(jpp.groupby("flight_date").size()
    .sort_values(ascending=False).head(10)
    .rename("nb_aeroports_touches").to_frame())

### La grève de juillet vue depuis Nice

In [ ]:
nice = stg[stg["airport_icao"] == "LFMN"].sort_values("flight_date")
nice_anom = jpp[jpp["airport_icao"] == "LFMN"]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(nice["flight_date"], nice["src1_total"], color=BLEU, linewidth=1.5, label="Vols par jour")
ax.scatter(nice_anom["flight_date"], nice_anom["src1_total"], s=50, color=ORANGE,
           edgecolor="#fcfcfb", linewidth=1.5, zorder=3, label="Jour perturbé (z-score < -3)")

greve = nice_anom[nice_anom["flight_date"] == pd.Timestamp("2025-07-04")].iloc[0]
ax.annotate(
    f"3-4 juillet : grève du contrôle aérien\n{int(greve['src1_total'])} vols au lieu de ~{int(greve['moy_voisins'])} attendus",
    xy=(greve["flight_date"], greve["src1_total"]),
    xytext=(pd.Timestamp("2025-03-01"), 180),
    arrowprops=dict(arrowstyle="->", color=ENCRE_2), color=ENCRE, fontsize=10,
)
ax.set_title("Nice-Côte d'Azur (LFMN) : trafic IFR quotidien en 2025")
ax.set_ylabel("Vols (départs + arrivées)")
ax.set_ylim(0, None)
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

**Lecture** : la méthode retrouve d'elle-même la grève du contrôle aérien des 3 et 4 juillet 2025, pour laquelle la DGAC avait demandé aux compagnies de réduire de 50 % leur programme à Nice.


## 4. Expliquer les perturbations

### 4.1 Annotation manuelle des événements

Les dates les plus touchées ont été recherchées une par une, puis consignées dans `data/evenements_2025_v2.csv` (date, pays, catégorie, événement, source). Cette table est jointe aux jours perturbés en SQL (`sql/05`), puis chaque anomalie est classée selon sa **portée** (`sql/06`) :
- **Réseau** : au moins 3 aéroports du même pays perturbés le même jour ;
- **Local** : 1 ou 2 aéroports.

> Leçon de qualité de données : la table d'événements doit utiliser **le même format** que la table qu'elle rejoint (dates `AAAA-MM-JJ`, noms de pays en anglais comme dans la source EUROCONTROL). Sinon la jointure ne trouve rien.

In [ ]:
synth = (jpp.groupby(["categorie", "portee"]).size().unstack(fill_value=0)
         .reindex(columns=["Réseau", "Local"], fill_value=0))
synth = synth.loc[synth.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(synth.index, synth["Réseau"], color=BLEU, height=0.6, label="Réseau (≥ 3 aéroports du pays)")
ax.barh(synth.index, synth["Local"], left=synth["Réseau"] + 2, color=ORANGE, height=0.6,
        label="Local (1-2 aéroports)")
for y, total in enumerate(synth.sum(axis=1)):
    ax.text(total + 8, y, str(total), va="center", color=ENCRE_2, fontsize=10)
ax.set_title("Jours-aéroport perturbés en 2025, par explication")
ax.set_xlabel("Nombre de jours-aéroport")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

synth.assign(total=synth.sum(axis=1)).sort_values("total", ascending=False)

**Lecture** : la deuxième vague d'enquête a fait passer les perturbations **réseau** non expliquées de 152 à 83. Les jours fériés pèsent lourd (Noël, Pâques en Norvège, Assomption…), ce qui est attendu. Les incidents **locaux** non expliqués (un seul aéroport, un jour) sont trop nombreux pour être enquêtés un par un : c'est une limite assumée.

### 4.2 Perturbations réseau encore non expliquées

Les prochaines pistes d'enquête :

In [ ]:
(jpp[(jpp["portee"] == "Réseau") & (jpp["categorie"] == "Non expliqué")]
    .groupby(["flight_date", "country"])
    .agg(nb_aeroports=("airport_icao", "size"),
         aeroports=("airport_name", lambda s: ", ".join(sorted(s))))
    .reset_index()
    .sort_values("flight_date"))

### 4.3 L'effet réseau de la grève du 3 juillet

Une grève en France affecte-t-elle les autres pays ? On compare le trafic de chaque pays le 3 juillet à celui des mêmes jours de semaine, ±4 semaines.

In [ ]:
reseau = lire(f"""
WITH pays_jour AS (
  SELECT country, flight_date, SUM(src1_total) AS total
  FROM `{PROJET}.{DATASET}.stg_airport_traffic`
  GROUP BY country, flight_date
)
SELECT
  jour.country,
  jour.total                                                      AS trafic_jour,
  AVG(voisin.total)                                               AS trafic_attendu,
  SAFE_DIVIDE(jour.total - AVG(voisin.total), AVG(voisin.total)) AS variation
FROM pays_jour AS jour
JOIN pays_jour AS voisin
  ON  voisin.country = jour.country
  AND ABS(DATE_DIFF(voisin.flight_date, jour.flight_date, DAY)) BETWEEN 7 AND 28
  AND MOD(DATE_DIFF(voisin.flight_date, jour.flight_date, DAY), 7) = 0
WHERE jour.flight_date = '2025-07-03'
GROUP BY jour.country, jour.total
HAVING AVG(voisin.total) >= 200          -- on écarte les pays à très faible trafic
ORDER BY variation
""")

d = reseau.sort_values("variation").reset_index(drop=True)
couleurs = [ORANGE if p == "France" else GRIS for p in d["country"]]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(d["country"], d["variation"] * 100, color=couleurs, height=0.7)
ax.invert_yaxis()   # le plus touché en haut
ax.axvline(0, color=ENCRE_2, linewidth=0.8)

# étiquettes sur les 5 pays les plus touchés seulement
for y, v in enumerate(d["variation"] * 100):
    if y == 0:      # barre longue : étiquette à l'intérieur, en blanc
        ax.text(v + 0.5, y, f"{v:.1f} %", va="center", ha="left", color="white", fontsize=10, fontweight="bold")
    elif y < 5:
        ax.text(v - 0.4, y, f"{v:.1f} %", va="center", ha="right", color=ENCRE, fontsize=10)

ax.set_title("3 juillet 2025 : écart au trafic attendu, par pays")
ax.set_xlabel("Variation vs mêmes jours de semaine, ±4 semaines (%)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Lecture** :
- la **France perd 32 %** de son trafic ce jour-là ;
- presque tous les autres pays baissent légèrement (−2 à −8 %), en tête le **Maroc, la Suisse et le Portugal**, des pays dont beaucoup de vols vont en France ou la survolent ;
- le Royaume-Uni (−4,5 %) n'est pas plus touché que la Norvège ou la Suède : on ne peut pas conclure à un effet spécifique.

👉 L'effet se **propage**, mais il reste **modeste et diffus**. On évite de surinterpréter un chiffre isolé.

## 5. Conclusions et limites

### Ce qu'il faut retenir

| Question | Réponse |
|---|---|
| Les deux sources sont-elles disponibles partout ? | Non : la source n°2 n'existe que pour 97 aéroports sur 331 |
| Sont-elles d'accord ? | Globalement oui (93 % des jours à moins de 5 % d'écart), avec quelques aéroports systématiquement divergents (Toulouse, Dresde, Bologne…) |
| La détection d'anomalies fonctionne-t-elle ? | Oui : elle retrouve seule la grève de juillet, la tempête Éowyn, la panne électrique ibérique, les grèves allemandes et belges |
| Quelle part est expliquée ? | La majorité des perturbations **réseau** ; les incidents **locaux** restent en grande partie à enquêter |

### Limites

- Données d'une seule année (2025) : pas de comparaison avec 2019 ni de tendance pluriannuelle.
- Le seuil z < −3 et la définition « réseau » (≥ 3 aéroports) sont des choix simples, à justifier ou à faire varier.
- L'annotation des événements est manuelle et non exhaustive.
- La définition exacte des deux sources reste à confirmer dans la documentation EUROCONTROL.

### Pistes

- Passer les fichiers SQL en **modèles dbt**, avec des tests de qualité automatiques.
- Détecter aussi les **pics** (z-score > +3) : grands événements, aviation d'affaires.
- Croiser avec les données de retard (ATFM) d'EUROCONTROL.

✍️ *À compléter : ta conclusion personnelle, en 3-4 phrases.*